# 01 · EDA, limpieza y análisis principal

## Proyecto: detonantes de frustración en clientes de call center / soporte

**Pregunta de análisis:**

> ¿Cuáles son los principales detonantes de frustración en clientes y cómo pueden utilizarse para diseñar escenarios de entrenamiento para agentes de soporte?

Este notebook documenta el proceso completo:

1. Carga del dataset original.
2. Exploración inicial de los datos, también llamada **EDA**.
3. Limpieza y preparación de texto.
4. Creación de nuevas variables útiles para análisis.
5. Análisis de sentimiento.
6. Clasificación de detonantes de frustración.
7. Análisis principal orientado a entrenamiento de agentes.
8. Exportación del archivo `dataset_procesado.csv`.

**Nota importante:**  
El dataset original TWCS es grande. Por eso el notebook incluye un modo rápido para trabajar con una muestra manejable en Google Colab. Si el equipo quiere procesar todo el archivo, puede desactivar ese modo.

## 0. Instalación e importación de librerías

En esta sección cargamos las librerías necesarias.

- `pandas`: permite trabajar con tablas de datos.
- `numpy`: ayuda con cálculos numéricos.
- `matplotlib`: permite crear gráficos.
- `re`: permite limpiar texto usando expresiones regulares.
- `nltk`: se usa para aplicar análisis de sentimiento con VADER.

VADER es un analizador de sentimiento útil para textos cortos como tweets porque reconoce palabras positivas, negativas, signos de exclamación, mayúsculas y expresiones comunes en redes sociales.

In [ ]:
# ============================================================
# 0. Instalación e importación de librerías
# ============================================================

# En Google Colab normalmente basta ejecutar esta celda.
# Si alguna librería ya está instalada, Colab simplemente la reutiliza.
!pip -q install nltk

import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Configuración visual básica para que las tablas se lean mejor.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Librerías cargadas correctamente.")

## 1. Configuración de rutas del proyecto

La entrega debería quedar ordenada así:

```text
proyectos/
    nombre-del-equipo/
        data/
            dataset_original.csv
            dataset_procesado.csv
        notebooks/
            01_eda_limpieza_analisis.ipynb
        outputs/
            top_detonantes.png
            sentimiento_por_detonante.png
        app.py
        requirements.txt
        README.md
```

Para trabajar en Colab, se puede subir el CSV directamente al entorno.  
Para trabajar en el repo final, lo ideal es renombrar el archivo original como:

```text
data/dataset_original.csv
```

Si el dataset original pesa más de 50 MB, no conviene subirlo al repositorio. En ese caso, en el `README.md` se debe poner un enlace de descarga y subir solo el dataset procesado si pesa poco.

In [ ]:
# ============================================================
# 1. Configuración de rutas
# ============================================================

# Carpeta raíz desde donde se ejecuta el notebook.
# En Colab suele ser /content.
PROJECT_ROOT = Path.cwd()

# Carpetas esperadas en la estructura final del proyecto.
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Creamos las carpetas si no existen.
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Posibles nombres del dataset original.
# Puedes agregar aquí el nombre exacto de tu archivo si lo cambias.
POSSIBLE_INPUTS = [
    DATA_DIR / "dataset_original.csv",
    DATA_DIR / "twcs.csv",
    DATA_DIR / "twcs(1).csv",
    PROJECT_ROOT / "dataset_original.csv",
    PROJECT_ROOT / "twcs.csv",
    PROJECT_ROOT / "twcs(1).csv",
]

INPUT_PATH = None

for path in POSSIBLE_INPUTS:
    if path.exists():
        INPUT_PATH = path
        break

# Si no se encontró el archivo en las rutas anteriores, buscamos cualquier CSV disponible.
if INPUT_PATH is None:
    csv_files = list(PROJECT_ROOT.glob("*.csv")) + list(DATA_DIR.glob("*.csv"))
    if len(csv_files) > 0:
        INPUT_PATH = csv_files[0]

# Si aún no se encuentra, se intenta pedir una carga manual en Google Colab.
if INPUT_PATH is None:
    try:
        from google.colab import files
        print("No se encontró un CSV. Sube el dataset original ahora.")
        uploaded = files.upload()
        uploaded_name = list(uploaded.keys())[0]
        INPUT_PATH = PROJECT_ROOT / uploaded_name
    except Exception:
        raise FileNotFoundError(
            "No se encontró el dataset. Sube el CSV a Colab o guárdalo como data/dataset_original.csv"
        )

print(f"Archivo seleccionado: {INPUT_PATH}")
print(f"Tamaño aproximado: {INPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB")

## 2. Carga del dataset

El dataset TWCS tiene conversaciones de soporte en Twitter. Las columnas principales son:

| Columna | Significado |
|---|---|
| `tweet_id` | Identificador único del tweet |
| `author_id` | Identificador del autor. Puede ser cliente o empresa |
| `inbound` | Indica si el mensaje viene desde el cliente hacia soporte |
| `created_at` | Fecha y hora del mensaje |
| `text` | Texto del mensaje |
| `response_tweet_id` | ID del tweet que respondió a este mensaje |
| `in_response_to_tweet_id` | ID del tweet al que este mensaje está respondiendo |

Para este proyecto nos interesa principalmente `text`, porque ahí está el reclamo, molestia, duda o comentario del cliente.

### Modo rápido

Como el archivo puede ser pesado, se usa `MODO_RAPIDO = True` para analizar una muestra.  
Esto permite desarrollar el proyecto sin que Colab se vuelva lento.

- Si quieres usar una muestra: deja `MODO_RAPIDO = True`.
- Si quieres procesar todo: cambia a `MODO_RAPIDO = False`.

Para una entrega académica, usar una muestra es válido si se declara en el notebook y en el README.

In [ ]:
# ============================================================
# 2. Carga del dataset
# ============================================================

# Cambia esto a False si quieres cargar todo el dataset.
# Advertencia: el archivo original puede pesar cerca de 500 MB o más.
MODO_RAPIDO = True

# Número máximo de filas para trabajar de forma cómoda en Colab.
# Puedes subirlo si tu sesión tiene suficiente memoria.
MAX_FILAS_ANALISIS = 200_000

# Columnas que sí necesitamos para el análisis.
# Cargar solo estas columnas reduce uso de memoria.
USECOLS = [
    "tweet_id",
    "author_id",
    "inbound",
    "created_at",
    "text",
    "response_tweet_id",
    "in_response_to_tweet_id",
]

def contar_filas_csv(path):
    """Cuenta filas del CSV para calcular una muestra aproximada.

    Restamos 1 porque la primera línea corresponde al encabezado.
    Esta función es útil cuando el archivo es grande y queremos muestrear
    sin cargarlo completo en memoria.
    """
    with open(path, "rb") as f:
        total = sum(1 for _ in f) - 1
    return max(total, 0)

def cargar_csv(path, modo_rapido=True, max_filas=200_000, chunksize=100_000):
    """Carga el dataset original.

    Si modo_rapido=True:
        - Calcula una fracción de muestreo.
        - Lee el archivo por partes.
        - Toma una muestra aleatoria de cada parte.
        - Devuelve una tabla manejable.

    Si modo_rapido=False:
        - Lee el archivo completo.
    """

    if not modo_rapido:
        df_full = pd.read_csv(
            path,
            usecols=USECOLS,
            low_memory=False
        )
        return df_full

    total_filas = contar_filas_csv(path)
    frac = min(1, max_filas / total_filas) if total_filas > 0 else 1

    print(f"Filas estimadas en archivo original: {total_filas:,}")
    print(f"Fracción aproximada de muestra: {frac:.4f}")

    partes = []
    rng_seed = 42

    for i, chunk in enumerate(pd.read_csv(path, usecols=USECOLS, chunksize=chunksize, low_memory=False)):
        # sample(frac=...) toma una muestra aleatoria de cada bloque.
        # random_state cambia por bloque para evitar repetir exactamente el mismo patrón.
        muestra_chunk = chunk.sample(frac=frac, random_state=rng_seed + i)
        partes.append(muestra_chunk)

    df_sample = pd.concat(partes, ignore_index=True)

    # Si por redondeos quedó una muestra mayor al máximo, reducimos al límite definido.
    if len(df_sample) > max_filas:
        df_sample = df_sample.sample(n=max_filas, random_state=42).reset_index(drop=True)

    return df_sample

df = cargar_csv(INPUT_PATH, modo_rapido=MODO_RAPIDO, max_filas=MAX_FILAS_ANALISIS)

print(f"Filas cargadas: {df.shape[0]:,}")
print(f"Columnas cargadas: {df.shape[1]:,}")
df.head()

## 3. Exploración inicial de datos, EDA

**EDA** significa *Exploratory Data Analysis* o análisis exploratorio de datos.

En esta parte buscamos entender el dataset antes de modificarlo:

- Cuántas filas y columnas tiene.
- Qué tipos de datos hay.
- Cuántos valores faltantes existen.
- Si hay duplicados.
- Qué proporción de mensajes son de clientes (`inbound=True`) versus empresas (`inbound=False`).

Esto es importante porque no conviene limpiar ni analizar sin conocer primero la estructura real del archivo.

In [ ]:
# ============================================================
# 3. EDA inicial
# ============================================================

print("Dimensiones del dataset:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

print("\nTipos de datos:")
display(df.dtypes)

print("\nPrimeras filas:")
display(df.head())

print("\nValores faltantes por columna:")
faltantes = (
    df.isna()
      .sum()
      .reset_index()
      .rename(columns={"index": "columna", 0: "valores_faltantes"})
)
faltantes["porcentaje_faltante"] = faltantes["valores_faltantes"] / len(df) * 100
display(faltantes.sort_values("porcentaje_faltante", ascending=False))

print("\nCantidad de filas duplicadas completas:")
print(df.duplicated().sum())

print("\nDistribución de inbound:")
display(df["inbound"].value_counts(dropna=False).rename_axis("inbound").reset_index(name="cantidad"))

### Interpretación de `inbound`

En este dataset:

- `inbound = True`: normalmente representa mensajes enviados por clientes hacia las cuentas de soporte.
- `inbound = False`: normalmente representa respuestas de empresas o agentes de soporte.

Como la pregunta del proyecto se centra en **frustración de clientes**, el análisis principal se hará sobre los mensajes `inbound=True`.

No eliminamos todavía las demás filas, porque pueden ayudar a identificar a qué empresa estaba dirigida la conversación.

In [ ]:
# ============================================================
# 3.1 Exploración de mensajes de clientes vs empresas
# ============================================================

conteo_inbound = df["inbound"].value_counts(dropna=False)

plt.figure(figsize=(6, 4))
conteo_inbound.plot(kind="bar")
plt.title("Cantidad de mensajes según inbound")
plt.xlabel("inbound")
plt.ylabel("Cantidad de mensajes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Limpieza y preparación de datos

En esta sección limpiamos el dataset.

Las decisiones principales son:

1. Convertir fechas a formato datetime.
2. Asegurar que `text` sea texto válido.
3. Eliminar filas sin texto.
4. Crear una versión limpia del texto.
5. Crear variables útiles como:
   - fecha
   - hora
   - día de semana
   - longitud del mensaje
   - cantidad de palabras

La columna `texto_limpio` no reemplaza al texto original.  
Se conserva `text` para trazabilidad y `texto_limpio` para análisis.

In [ ]:
# ============================================================
# 4. Limpieza de fechas y texto
# ============================================================

df_limpio = df.copy()

# Convertimos tweet_id a texto para evitar problemas si hay valores grandes o IDs con formato mixto.
df_limpio["tweet_id"] = df_limpio["tweet_id"].astype(str)

# Convertimos fechas.
# errors="coerce" transforma fechas inválidas en NaT, en vez de romper el notebook.
df_limpio["created_at"] = pd.to_datetime(df_limpio["created_at"], errors="coerce", utc=True)

# Aseguramos que text sea texto y eliminamos filas sin contenido.
df_limpio["text"] = df_limpio["text"].astype(str)
df_limpio = df_limpio[df_limpio["text"].str.strip().ne("")]
df_limpio = df_limpio[df_limpio["text"].str.lower().ne("nan")]

# Variables temporales útiles para el dashboard.
df_limpio["fecha"] = df_limpio["created_at"].dt.date
df_limpio["hora"] = df_limpio["created_at"].dt.hour
df_limpio["dia_semana"] = df_limpio["created_at"].dt.day_name()

# Variables de estructura del texto.
df_limpio["longitud_texto"] = df_limpio["text"].str.len()
df_limpio["num_palabras"] = df_limpio["text"].str.split().str.len()

# Variables binarias simples.
df_limpio["tiene_url"] = df_limpio["text"].str.contains(r"http\S+|www\.\S+", regex=True, na=False)
df_limpio["tiene_mencion"] = df_limpio["text"].str.contains(r"@\w+", regex=True, na=False)

print(f"Filas después de limpieza básica: {len(df_limpio):,}")
df_limpio.head()

In [ ]:
# ============================================================
# 4.1 Función para limpiar texto
# ============================================================

def limpiar_texto(texto):
    """Limpia texto para análisis.

    Esta función:
    - pasa el texto a minúsculas;
    - elimina URLs;
    - elimina menciones de Twitter;
    - elimina caracteres que no aportan mucho al análisis;
    - reduce espacios repetidos.

    No buscamos una limpieza extrema porque VADER puede aprovechar signos
    como ! o palabras completas para detectar emoción.
    """
    texto = str(texto).lower()
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)        # elimina URLs
    texto = re.sub(r"@\w+", " ", texto)                     # elimina menciones
    texto = re.sub(r"&amp;", " and ", texto)                 # normaliza símbolo frecuente en Twitter
    texto = re.sub(r"[^a-záéíóúñü0-9\s!?.']", " ", texto)   # conserva letras, números y algunos signos
    texto = re.sub(r"\s+", " ", texto).strip()              # elimina espacios repetidos
    return texto

df_limpio["texto_limpio"] = df_limpio["text"].apply(limpiar_texto)

display(df_limpio[["text", "texto_limpio"]].head(10))

## 5. Identificación aproximada de empresa contactada

En TWCS, algunas filas tienen `author_id` con nombres de empresas, por ejemplo cuentas de soporte.  
Los clientes suelen aparecer como IDs numéricos.

Esta identificación no es perfecta, pero ayuda a crear un filtro útil para el dashboard.

Estrategia usada:

1. Si el mensaje del cliente responde a un tweet de una empresa, tomamos esa empresa.
2. Si una empresa respondió al mensaje del cliente, tomamos esa empresa.
3. Si no se puede identificar, dejamos `No identificada`.

Esta variable se llama `empresa_contactada`.

In [ ]:
# ============================================================
# 5. Identificación aproximada de empresa contactada
# ============================================================

def extraer_primer_id(valor):
    """Extrae el primer ID desde campos que pueden venir como número, texto o lista separada por comas.

    Ejemplos:
    - 123 -> "123"
    - "123,456" -> "123"
    - NaN -> None
    """
    if pd.isna(valor):
        return None

    valor = str(valor).strip()

    if valor == "" or valor.lower() == "nan":
        return None

    # Algunos IDs vienen como "123.0" por conversión automática.
    if valor.endswith(".0"):
        valor = valor[:-2]

    # Algunos campos pueden tener varios IDs separados por coma.
    if "," in valor:
        valor = valor.split(",")[0].strip()

    return valor

# Normalizamos IDs.
df_limpio["tweet_id_norm"] = df_limpio["tweet_id"].apply(extraer_primer_id)
df_limpio["in_response_to_norm"] = df_limpio["in_response_to_tweet_id"].apply(extraer_primer_id)
df_limpio["response_first_norm"] = df_limpio["response_tweet_id"].apply(extraer_primer_id)

# Una cuenta de empresa suele tener un author_id no numérico.
df_limpio["author_id_str"] = df_limpio["author_id"].astype(str)
df_limpio["autor_es_empresa"] = ~df_limpio["author_id_str"].str.fullmatch(r"\d+")

# Mapa tweet_id -> author_id para saber quién escribió el tweet relacionado.
mapa_autor_por_tweet = df_limpio.set_index("tweet_id_norm")["author_id_str"].to_dict()

df_limpio["autor_tweet_respondido"] = df_limpio["in_response_to_norm"].map(mapa_autor_por_tweet)
df_limpio["autor_tweet_respuesta"] = df_limpio["response_first_norm"].map(mapa_autor_por_tweet)

def elegir_empresa(row):
    """Elige una empresa probable asociada al mensaje.

    Se considera empresa cuando el author_id no es puramente numérico.
    """
    candidatos = [
        row.get("autor_tweet_respondido"),
        row.get("autor_tweet_respuesta"),
        row.get("author_id_str") if row.get("autor_es_empresa") else None,
    ]

    for candidato in candidatos:
        if candidato is None or pd.isna(candidato):
            continue
        candidato = str(candidato)
        if not re.fullmatch(r"\d+", candidato):
            return candidato

    return "No identificada"

df_limpio["empresa_contactada"] = df_limpio.apply(elegir_empresa, axis=1)

print("Empresas/contactos más frecuentes en la muestra:")
display(
    df_limpio["empresa_contactada"]
    .value_counts()
    .head(15)
    .rename_axis("empresa_contactada")
    .reset_index(name="cantidad")
)

## 6. Filtro principal: mensajes de clientes

La pregunta del proyecto se enfoca en frustración de clientes.  
Por eso, desde aquí trabajamos con mensajes `inbound=True`.

Esto reduce ruido, porque las respuestas de soporte pueden tener lenguaje positivo o protocolar, y eso podría distorsionar el análisis de frustración.

In [ ]:
# ============================================================
# 6. Dataset principal: solo mensajes inbound de clientes
# ============================================================

df_clientes = df_limpio[df_limpio["inbound"] == True].copy()

print(f"Mensajes totales en muestra limpia: {len(df_limpio):,}")
print(f"Mensajes de clientes inbound=True: {len(df_clientes):,}")
print(f"Porcentaje de mensajes de clientes: {len(df_clientes) / len(df_limpio) * 100:.2f}%")

df_clientes[["tweet_id", "created_at", "empresa_contactada", "text", "texto_limpio"]].head()

## 7. Análisis de sentimiento

El análisis de sentimiento clasifica cada mensaje según su tono emocional.

Usaremos tres etiquetas:

| Etiqueta | Criterio |
|---|---|
| `positivo` | score mayor o igual a 0.25 |
| `neutral` | score entre -0.25 y 0.25 |
| `negativo` | score menor o igual a -0.25 |

El score viene de VADER y va aproximadamente desde -1 a 1:

- Cerca de -1: muy negativo.
- Cerca de 0: neutro.
- Cerca de 1: muy positivo.

Para este proyecto, los mensajes negativos son especialmente importantes porque pueden revelar frustración, enojo, espera, mala experiencia o problemas no resueltos.

In [ ]:
# ============================================================
# 7. Análisis de sentimiento con VADER
# ============================================================

import nltk
nltk.download("vader_lexicon")

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def obtener_sentimiento_score(texto):
    """Calcula el score de sentimiento usando VADER.

    Devuelve el valor compound, que resume el sentimiento general del texto.
    """
    return sia.polarity_scores(str(texto))["compound"]

def clasificar_sentimiento(score):
    """Convierte el score numérico en una etiqueta fácil de interpretar."""
    if score <= -0.25:
        return "negativo"
    elif score >= 0.25:
        return "positivo"
    else:
        return "neutral"

def clasificar_intensidad(score):
    """Entrega una lectura más detallada del nivel emocional."""
    if score <= -0.60:
        return "muy negativo"
    elif score <= -0.25:
        return "negativo"
    elif score < 0.25:
        return "neutral"
    elif score < 0.60:
        return "positivo"
    else:
        return "muy positivo"

df_clientes["sentimiento_score"] = df_clientes["texto_limpio"].apply(obtener_sentimiento_score)
df_clientes["sentimiento"] = df_clientes["sentimiento_score"].apply(clasificar_sentimiento)
df_clientes["intensidad_sentimiento"] = df_clientes["sentimiento_score"].apply(clasificar_intensidad)

print("Distribución de sentimiento:")
display(
    df_clientes["sentimiento"]
    .value_counts(normalize=False)
    .rename_axis("sentimiento")
    .reset_index(name="cantidad")
)

display(
    df_clientes[["text", "texto_limpio", "sentimiento_score", "sentimiento", "intensidad_sentimiento"]]
    .sample(min(10, len(df_clientes)), random_state=42)
)

In [ ]:
# ============================================================
# 7.1 Gráfico de distribución de sentimiento
# ============================================================

sent_counts = df_clientes["sentimiento"].value_counts()

plt.figure(figsize=(7, 4))
sent_counts.plot(kind="bar")
plt.title("Distribución de sentimiento en mensajes de clientes")
plt.xlabel("Sentimiento")
plt.ylabel("Cantidad de mensajes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Clasificación de detonantes de frustración

Un **detonante de frustración** es la causa probable que genera molestia en el cliente.

Ejemplos:

- Espera demasiado larga.
- Cobro incorrecto.
- Problema con reembolso.
- Mala atención de un agente.
- Producto o servicio que no funciona.
- Pedido perdido o retrasado.
- Falta de respuesta.

En este notebook se usa una clasificación por reglas y palabras clave.  
No es perfecta, pero es transparente, explicable y suficiente para construir un primer dashboard.

La ventaja de este método es que podemos justificar por qué un mensaje cayó en una categoría.

In [ ]:
# ============================================================
# 8. Diccionario de detonantes de frustración
# ============================================================

# Cada categoría tiene palabras o patrones asociados.
# Incluye inglés y algo de español por seguridad, aunque TWCS es principalmente inglés.
DETONANTES = {
    "demora_espera": [
        r"\bwait\b", r"\bwaiting\b", r"\bwaited\b", r"\bhours?\b", r"\bdays?\b",
        r"\blate\b", r"\bdelay\b", r"\bdelayed\b", r"\bstill waiting\b",
        r"\bno update\b", r"\btaking forever\b", r"\bdemora\b", r"\bespera\b"
    ],

    "cobro_pago_reembolso": [
        r"\bcharge\b", r"\bcharged\b", r"\bcharging\b", r"\bbill\b", r"\bbilling\b",
        r"\brefund\b", r"\brefunded\b", r"\bpayment\b", r"\bpaid\b",
        r"\bfee\b", r"\bfees\b", r"\bcard\b", r"\bmoney\b",
        r"\bovercharged\b", r"\bcobro\b", r"\bpago\b", r"\breembolso\b"
    ],

    "problema_tecnico_app_web": [
        r"\bapp\b", r"\bwebsite\b", r"\bsite\b", r"\blogin\b", r"\bsign in\b",
        r"\berror\b", r"\bbug\b", r"\bcrash\b", r"\bdown\b",
        r"\bnot working\b", r"\bdoesn't work\b", r"\bcant access\b", r"\bcan't access\b",
        r"\bproblema técnico\b", r"\bapp\b", r"\bweb\b"
    ],

    "mala_atencion_agente": [
        r"\brude\b", r"\bunhelpful\b", r"\bterrible service\b", r"\bbad service\b",
        r"\bcustomer service\b", r"\bagent\b", r"\brepresentative\b", r"\bsupport\b",
        r"\bhung up\b", r"\bignored\b", r"\bno help\b",
        r"\bmala atención\b", r"\bejecutivo\b", r"\bagente\b"
    ],

    "pedido_envio_entrega": [
        r"\border\b", r"\bpackage\b", r"\bdelivery\b", r"\bdelivered\b",
        r"\bshipping\b", r"\bshipment\b", r"\btracking\b", r"\blost\b",
        r"\bmissing\b", r"\bwhere is my\b", r"\bnever arrived\b",
        r"\bpedido\b", r"\benvío\b", r"\bentrega\b"
    ],

    "cancelacion_reserva": [
        r"\bcancel\b", r"\bcancelled\b", r"\bcanceled\b", r"\bcancellation\b",
        r"\breservation\b", r"\bbooking\b", r"\bflight\b", r"\btrip\b",
        r"\bhotel\b", r"\breserva\b", r"\bcancelación\b"
    ],

    "servicio_caido_calidad": [
        r"\boutage\b", r"\bsignal\b", r"\binternet\b", r"\bconnection\b",
        r"\bservice down\b", r"\bno service\b", r"\bbroken\b",
        r"\bdoes not work\b", r"\bstopped working\b", r"\bquality\b",
        r"\bservicio caído\b", r"\bsin servicio\b"
    ],

    "falta_respuesta_comunicacion": [
        r"\bno response\b", r"\bno reply\b", r"\bemail\b", r"\bcalled\b",
        r"\bcall\b", r"\bphone\b", r"\bcontacted\b", r"\btold me\b",
        r"\bnever heard\b", r"\bsin respuesta\b", r"\bno contestan\b"
    ],
}

ESCENARIOS_ENTRENAMIENTO = {
    "demora_espera": "Manejo de espera: reconocer molestia, explicar causa, entregar plazo y próximo paso.",
    "cobro_pago_reembolso": "Caso de cobro o reembolso: validar datos, explicar proceso y cerrar con compromiso verificable.",
    "problema_tecnico_app_web": "Soporte técnico: guiar diagnóstico paso a paso sin culpar al cliente.",
    "mala_atencion_agente": "Recuperación de experiencia: disculpa concreta, escucha activa y reparación del vínculo.",
    "pedido_envio_entrega": "Seguimiento de pedido: confirmar estado, explicar retraso y ofrecer alternativa.",
    "cancelacion_reserva": "Cancelación o reserva: explicar política, revisar excepciones y proponer opciones.",
    "servicio_caido_calidad": "Servicio caído o mala calidad: contener frustración y entregar plan de acción.",
    "falta_respuesta_comunicacion": "Falta de respuesta: asumir quiebre comunicacional y ordenar un canal único de seguimiento.",
    "otro_no_clasificado": "Caso general: escuchar, resumir problema, confirmar expectativa y escalar si corresponde.",
}

def clasificar_detonante(texto):
    """Clasifica el detonante principal del mensaje.

    La función revisa cada categoría y cuenta cuántos patrones aparecen.
    Devuelve:
    - detonante con mayor cantidad de coincidencias;
    - cantidad de coincidencias encontradas.

    Si no encuentra ninguna coincidencia, marca otro_no_clasificado.
    """
    texto = str(texto).lower()

    puntajes = {}

    for categoria, patrones in DETONANTES.items():
        puntaje = 0
        for patron in patrones:
            if re.search(patron, texto):
                puntaje += 1
        puntajes[categoria] = puntaje

    mejor_categoria = max(puntajes, key=puntajes.get)
    mejor_puntaje = puntajes[mejor_categoria]

    if mejor_puntaje == 0:
        return "otro_no_clasificado", 0

    return mejor_categoria, mejor_puntaje

resultados_detonante = df_clientes["texto_limpio"].apply(clasificar_detonante)

df_clientes["detonante_frustracion"] = resultados_detonante.apply(lambda x: x[0])
df_clientes["detonante_score"] = resultados_detonante.apply(lambda x: x[1])
df_clientes["escenario_sugerido"] = df_clientes["detonante_frustracion"].map(ESCENARIOS_ENTRENAMIENTO)

display(
    df_clientes[["text", "sentimiento", "sentimiento_score", "detonante_frustracion", "detonante_score", "escenario_sugerido"]]
    .sample(min(10, len(df_clientes)), random_state=42)
)

## 9. Variable de prioridad

Para convertir el análisis en algo útil para entrenamiento, creamos una variable llamada `prioridad_entrenamiento`.

La lógica es simple:

| Prioridad | Criterio |
|---|---|
| `alta` | Mensaje muy negativo o negativo con detonante claro |
| `media` | Mensaje negativo o con algún detonante |
| `baja` | Mensaje positivo, neutral o sin señal clara de frustración |

Esto permite que el dashboard filtre rápidamente los casos más útiles para entrenar agentes.

In [ ]:
# ============================================================
# 9. Creación de prioridad de entrenamiento
# ============================================================

def asignar_prioridad(row):
    """Asigna prioridad para entrenamiento de agentes.

    La prioridad alta apunta a mensajes que probablemente representan
    interacciones difíciles o casos críticos.
    """
    score = row["sentimiento_score"]
    detonante_score = row["detonante_score"]
    sentimiento = row["sentimiento"]

    if score <= -0.60:
        return "alta"

    if sentimiento == "negativo" and detonante_score >= 1:
        return "alta"

    if sentimiento == "negativo" or detonante_score >= 1:
        return "media"

    return "baja"

df_clientes["prioridad_entrenamiento"] = df_clientes.apply(asignar_prioridad, axis=1)

display(
    df_clientes["prioridad_entrenamiento"]
    .value_counts()
    .rename_axis("prioridad_entrenamiento")
    .reset_index(name="cantidad")
)

## 10. Análisis principal

Ahora respondemos la pregunta del proyecto con tablas y gráficos.

Buscamos:

1. Qué detonantes aparecen con mayor frecuencia.
2. Qué detonantes tienen mayor proporción de sentimiento negativo.
3. Qué casos deberían usarse como escenarios de entrenamiento.
4. Qué ejemplos reales pueden alimentar el chatbot entrenador.

In [ ]:
# ============================================================
# 10. Resumen por detonante
# ============================================================

resumen_detonantes = (
    df_clientes
    .groupby("detonante_frustracion")
    .agg(
        mensajes=("tweet_id", "count"),
        sentimiento_promedio=("sentimiento_score", "mean"),
        pct_negativo=("sentimiento", lambda x: (x == "negativo").mean() * 100),
        pct_prioridad_alta=("prioridad_entrenamiento", lambda x: (x == "alta").mean() * 100),
        palabras_promedio=("num_palabras", "mean")
    )
    .reset_index()
    .sort_values(["mensajes", "pct_negativo"], ascending=[False, False])
)

display(resumen_detonantes)

# Guardamos esta tabla como CSV auxiliar para documentación.
resumen_detonantes.to_csv(OUTPUT_DIR / "resumen_detonantes.csv", index=False)

In [ ]:
# ============================================================
# 10.1 Gráfico: principales detonantes
# ============================================================

top_detonantes = resumen_detonantes.head(10).sort_values("mensajes", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(top_detonantes["detonante_frustracion"], top_detonantes["mensajes"])
plt.title("Top 10 detonantes de frustración por cantidad de mensajes")
plt.xlabel("Cantidad de mensajes")
plt.ylabel("Detonante")
plt.tight_layout()

# Se guarda el gráfico para la carpeta outputs.
plt.savefig(OUTPUT_DIR / "top_detonantes.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 10.2 Gráfico: porcentaje negativo por detonante
# ============================================================

neg_detonantes = resumen_detonantes.sort_values("pct_negativo", ascending=True).tail(10)

plt.figure(figsize=(9, 5))
plt.barh(neg_detonantes["detonante_frustracion"], neg_detonantes["pct_negativo"])
plt.title("Detonantes con mayor proporción de sentimiento negativo")
plt.xlabel("% de mensajes negativos")
plt.ylabel("Detonante")
plt.tight_layout()

plt.savefig(OUTPUT_DIR / "sentimiento_por_detonante.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 10.3 Tabla cruzada: detonante vs sentimiento
# ============================================================

tabla_sentimiento_detonante = pd.crosstab(
    df_clientes["detonante_frustracion"],
    df_clientes["sentimiento"],
    normalize="index"
) * 100

tabla_sentimiento_detonante = tabla_sentimiento_detonante.round(2)

display(tabla_sentimiento_detonante)

tabla_sentimiento_detonante.to_csv(OUTPUT_DIR / "tabla_sentimiento_detonante.csv")

## 11. Ejemplos reales para escenarios de entrenamiento

Esta sección extrae ejemplos de mensajes con prioridad alta.

Estos ejemplos pueden utilizarse después en el chatbot entrenador o en la presentación final.

La idea no es mostrar datos sensibles, sino usar los textos como casos de práctica:

- El agente debe reconocer la emoción.
- Identificar el detonante.
- Responder con empatía.
- Proponer una acción concreta.
- Cerrar con un próximo paso verificable.

In [ ]:
# ============================================================
# 11. Casos de entrenamiento sugeridos
# ============================================================

casos_entrenamiento = (
    df_clientes[df_clientes["prioridad_entrenamiento"] == "alta"]
    .sort_values("sentimiento_score", ascending=True)
    [[
        "tweet_id",
        "empresa_contactada",
        "text",
        "sentimiento_score",
        "sentimiento",
        "detonante_frustracion",
        "escenario_sugerido",
        "prioridad_entrenamiento"
    ]]
    .head(30)
)

display(casos_entrenamiento)

# Guardamos casos útiles para alimentar el chatbot o documentar el proyecto.
casos_entrenamiento.to_csv(OUTPUT_DIR / "casos_entrenamiento.csv", index=False)

## 12. Conclusiones preliminares

Esta celda genera conclusiones automáticas básicas usando los resultados del análisis.

Estas conclusiones se pueden copiar al README, a la presentación o al dashboard.

No reemplazan la interpretación humana, pero ayudan a ordenar la historia del proyecto.

In [ ]:
# ============================================================
# 12. Conclusiones automáticas preliminares
# ============================================================

total_clientes = len(df_clientes)
pct_negativo_total = (df_clientes["sentimiento"] == "negativo").mean() * 100
pct_alta_total = (df_clientes["prioridad_entrenamiento"] == "alta").mean() * 100

detonante_top = resumen_detonantes.iloc[0]["detonante_frustracion"]
detonante_top_mensajes = int(resumen_detonantes.iloc[0]["mensajes"])

detonante_mas_negativo = resumen_detonantes.sort_values("pct_negativo", ascending=False).iloc[0]["detonante_frustracion"]
pct_mas_negativo = resumen_detonantes.sort_values("pct_negativo", ascending=False).iloc[0]["pct_negativo"]

print("CONCLUSIONES PRELIMINARES")
print("-" * 60)
print(f"1. Se analizaron {total_clientes:,} mensajes de clientes.")
print(f"2. El {pct_negativo_total:.2f}% de los mensajes fue clasificado como negativo.")
print(f"3. El {pct_alta_total:.2f}% de los mensajes quedó como prioridad alta para entrenamiento.")
print(f"4. El detonante más frecuente fue '{detonante_top}', con {detonante_top_mensajes:,} mensajes.")
print(f"5. El detonante con mayor proporción de mensajes negativos fue '{detonante_mas_negativo}', con {pct_mas_negativo:.2f}% de negatividad.")
print("\nInterpretación:")
print("Los detonantes más frecuentes y más negativos deberían transformarse en escenarios de entrenamiento para que los agentes practiquen contención emocional, diagnóstico del problema y cierre con compromisos claros.")

## 13. Exportación del dataset procesado

Esta es la parte clave para la entrega.

El archivo `dataset_procesado.csv` será usado por `app.py` en Streamlit.

Se recomienda que el dashboard no limpie todo desde cero cada vez.  
El flujo correcto es:

1. Notebook procesa el dataset original.
2. Notebook exporta `dataset_procesado.csv`.
3. Streamlit carga `dataset_procesado.csv`.
4. Dashboard muestra KPIs, filtros, gráficos y ejemplos.

Columnas principales del dataset procesado:

| Columna | Uso |
|---|---|
| `text` | Texto original del cliente |
| `texto_limpio` | Texto limpio para análisis |
| `sentimiento` | Positivo, neutral o negativo |
| `sentimiento_score` | Puntaje numérico del sentimiento |
| `detonante_frustracion` | Causa probable de frustración |
| `escenario_sugerido` | Cómo convertir el caso en entrenamiento |
| `prioridad_entrenamiento` | Baja, media o alta |
| `empresa_contactada` | Empresa aproximada asociada a la conversación |

In [ ]:
# ============================================================
# 13. Exportar dataset procesado
# ============================================================

columnas_finales = [
    "tweet_id",
    "created_at",
    "fecha",
    "hora",
    "dia_semana",
    "author_id",
    "empresa_contactada",
    "text",
    "texto_limpio",
    "sentimiento_score",
    "sentimiento",
    "intensidad_sentimiento",
    "detonante_frustracion",
    "detonante_score",
    "escenario_sugerido",
    "prioridad_entrenamiento",
    "longitud_texto",
    "num_palabras",
    "tiene_url",
    "tiene_mencion",
    "response_tweet_id",
    "in_response_to_tweet_id",
]

# Algunas columnas pueden no existir si se modifica el notebook.
# Esta línea evita errores si falta alguna.
columnas_finales = [col for col in columnas_finales if col in df_clientes.columns]

df_procesado = df_clientes[columnas_finales].copy()

# Ordenamos por fecha cuando exista.
if "created_at" in df_procesado.columns:
    df_procesado = df_procesado.sort_values("created_at")

# Ruta final del CSV procesado.
OUTPUT_CSV = DATA_DIR / "dataset_procesado.csv"

df_procesado.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Dataset procesado exportado correctamente en: {OUTPUT_CSV}")
print(f"Filas exportadas: {len(df_procesado):,}")
print(f"Columnas exportadas: {len(df_procesado.columns):,}")

display(df_procesado.head())

## 14. Validación rápida del archivo exportado

Antes de cerrar el notebook, verificamos que el CSV procesado:

- exista;
- pueda leerse correctamente;
- tenga las columnas necesarias para el dashboard.

In [ ]:
# ============================================================
# 14. Validación final
# ============================================================

assert OUTPUT_CSV.exists(), "No se encontró el archivo dataset_procesado.csv"

df_test = pd.read_csv(OUTPUT_CSV)

columnas_necesarias_dashboard = [
    "text",
    "sentimiento",
    "sentimiento_score",
    "detonante_frustracion",
    "escenario_sugerido",
    "prioridad_entrenamiento"
]

faltantes_dashboard = [col for col in columnas_necesarias_dashboard if col not in df_test.columns]

if len(faltantes_dashboard) == 0:
    print("Validación exitosa: el dataset procesado tiene las columnas necesarias para Streamlit.")
else:
    print("Faltan columnas importantes para el dashboard:")
    print(faltantes_dashboard)

print("\nVista rápida del dataset procesado:")
display(df_test.head())

print("\nTamaño del archivo procesado:")
print(f"{OUTPUT_CSV.stat().st_size / 1024 / 1024:.2f} MB")

# Cierre del notebook

Con este notebook ya queda documentado el proceso de EDA, limpieza y análisis.

Archivos generados:

```text
data/dataset_procesado.csv
outputs/resumen_detonantes.csv
outputs/casos_entrenamiento.csv
outputs/top_detonantes.png
outputs/sentimiento_por_detonante.png
outputs/tabla_sentimiento_detonante.csv
```

El siguiente paso es construir `app.py` en Streamlit usando `data/dataset_procesado.csv`.

Recomendación para el dashboard:

- KPI 1: total de mensajes analizados.
- KPI 2: porcentaje de mensajes negativos.
- KPI 3: cantidad de casos de prioridad alta.
- Gráfico 1: top detonantes.
- Gráfico 2: sentimiento por detonante.
- Gráfico 3: casos por prioridad.
- Filtros: sentimiento, detonante, empresa y prioridad.
- Sección final: ejemplos reales para entrenamiento.